# B6 — Information environment and informed retail flow

Does the information environment around an announcement change how
informative retail option flow is? This connects the option-market tests
(B4) back to the thesis's central variable, analyst forecast dispersion.
The thesis finds dispersion does not change retail *behaviour* once firm
size is controlled; this asks whether it changes how *informative* that
behaviour is.

**Hypotheses stated before any result was seen** (this notebook was committed
before it was run):

| | Hypothesis | Prediction |
|---|---|---|
| **D1 (primary)** | Retail flow is more informative when **analyst dispersion** is high: public information is weakest there, so private information is most valuable | **flow × dispersion < 0**, linearly and for the top quartile (the thesis's dispersion effects are confined to the top quartile) |
| **D2** | Retail flow is more informative when **analyst coverage** is thin | **flow × coverage > 0** (the flow effect weakens as coverage rises); coverage = log number of analysts |
| **D3** | The same with **prior earnings volatility**, the thesis's second uncertainty measure | **flow × earnings volatility < 0** |

**The critical control is firm size.** Dispersion is strongly correlated with
firm size, so a flow × dispersion interaction alone could simply pick up
small firms. Every specification includes **flow × size** and **flow ×
attention** (B5 found flow more informative among low-attention events).
The version without the size interaction is reported as descriptive, to
show how much size was doing. On synthetic data where only size mattered,
omitting the size interaction produced a highly significant but spurious
dispersion effect; including it removed the effect entirely.

**Reading the results.** Judge each hypothesis by whether its forms and
outcomes agree. The `flow_z:mod` row is the test; `flow_z:size_z` and
`flow_z:attn_z` are the controls' interactions.

**Requires A1–A3.**

## Setup

In [ ]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl

## Build the event panel

The sector panel from B5 carries everything needed: the Test 1 variables,
analyst coverage, attention and abnormal flow. The contract panel is not
needed here.

In [ ]:
from analysis.informed_trading import build_event_panel
from analysis.sector_analysis import build_sector_panel

panel = build_event_panel(verbose=False)
sp = build_sector_panel(panel)

## Why size must be controlled

Correlations between the moderators and firm size across events. A strong
negative correlation between dispersion and size is why every
specification includes flow × size.

In [ ]:
chk = sp.select("dispersion_scaled", "earnings_volatility", "NUMEST", "log_mktcap").drop_nulls()
chk = chk.with_columns(pl.col("NUMEST").cast(pl.Float64).log().alias("coverage")).drop("NUMEST")
print(pl.DataFrame({"with_log_mktcap": [c for c in chk.columns if c != "log_mktcap"],
                    "correlation": [chk.select(pl.corr(c, "log_mktcap")).item()
                                    for c in chk.columns if c != "log_mktcap"]}))

## D1–D3

All pre-stated tests, for announcement returns (`car01`) and earnings
surprises (`surprise_scaled`), with the robustness rows: abnormal flow,
professional flow as a comparison, and D1 without the size interaction.

In [ ]:
from analysis.information_environment import test_information_environment

ie = test_information_environment(sp)
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=220, fmt_str_lengths=50):
    print(ie.select("hypothesis", "test", "outcome", "term", "n", "coef", "se", "p"))

## Secondary: by era

Retail flow was informative only before COVID (Test 1), so if dispersion
moderates it anywhere, the pre-COVID era is where it should show. Samples
are smaller; read for direction.

In [ ]:
from analysis.informed_trading import ERAS
from analysis.information_environment import moderator_test

rows = []
for e in ERAS:
    sub = sp.filter(pl.col("era") == e)
    for form in ("linear", "top_quartile"):
        for r in moderator_test(sub, "dispersion", "car01", form=form):
            rows.append({**r, "era": e})
with pl.Config(tbl_rows=-1, float_precision=4, tbl_width_chars=200, fmt_str_lengths=40):
    print(pl.DataFrame(rows).filter(pl.col("term").is_in(["flow_z", "flow_z:mod"]))
          .select("era", "test", "term", "n", "coef", "p"))

## Write results

`table14_information_environment.csv` to `results/`.

In [ ]:
from analysis.information_environment import write_information_results

ie_results = write_information_results(sp)